# 03 Model Validation and Calibration

This notebook evaluates the existing interpretable logistic regression baseline using training-set cross-validation, one held-out test evaluation, calibration assessment, and exploratory threshold analysis. It is an educational workflow and not clinical validation.

## Methodological Boundaries

- The original UCI target is preserved.
- `target_binary` maps original `0` to `0` and values above `0` to `1`.
- The held-out test set is separated before cross-validation.
- Cross-validation is performed only on training data.
- All preprocessing remains inside the scikit-learn pipeline.
- No threshold is proposed for clinical use.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data_processing import (  # noqa: E402
    HEART_DISEASE_COLUMNS,
    binarize_heart_disease_target,
    clean_missing_values,
    load_heart_disease_data,
)
from src.evaluation import (  # noqa: E402
    compute_classification_metrics,
    plot_calibration_assessment,
    plot_confusion_matrix,
    plot_cross_validation_metrics,
    plot_threshold_metrics,
    summarize_cross_validation,
    threshold_metrics_table,
)
from src.features import (  # noqa: E402
    define_heart_disease_feature_columns,
    feature_column_names,
)
from src.modeling import (  # noqa: E402
    create_logistic_regression_pipeline,
    split_train_test,
)

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "processed.cleveland.data"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## Load and Define the Analysis Data

In [2]:
if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(
        "Expected data/raw/processed.cleveland.data. Download the processed Cleveland file from the official UCI Heart Disease dataset page."
    )

df = clean_missing_values(load_heart_disease_data(RAW_DATA_PATH))
for column in HEART_DISEASE_COLUMNS:
    df[column] = pd.to_numeric(df[column], errors="coerce")
df["target_binary"] = binarize_heart_disease_target(df, "target")["target"]

feature_columns = define_heart_disease_feature_columns()
feature_names = feature_column_names(feature_columns)
x = df[feature_names].copy()
y = df["target_binary"].copy()

print(f"Rows: {len(df)}")
print(f"Features: {len(feature_names)}")
print(df[["target", "target_binary"]].head().to_string(index=False))

Rows: 303
Features: 13
 target  target_binary
      0              0
      2              1
      1              1
      0              0
      0              0


## Held-Out Test Design

A stratified 80/20 split is created once. The test set is not used during cross-validation or preprocessing fitting.

In [3]:
x_train, x_test, y_train, y_test = split_train_test(
    x,
    y,
    test_size=0.2,
    random_state=42,
)

print(f"Training rows: {len(x_train)}")
print(f"Held-out test rows: {len(x_test)}")
print("Training target proportions")
print(y_train.value_counts(normalize=True).sort_index().round(3).to_string())
print("Held-out target proportions")
print(y_test.value_counts(normalize=True).sort_index().round(3).to_string())

Training rows: 242
Held-out test rows: 61
Training target proportions
target_binary
0    0.541
1    0.459
Held-out target proportions
target_binary
0    0.541
1    0.459


## Training-Set Cross-Validation

Five-fold stratified cross-validation estimates variation across training folds. The full preprocessing and logistic regression pipeline is refitted independently within each fold.

In [4]:
validation_model = create_logistic_regression_pipeline(
    feature_columns,
    random_state=42,
)
cv_summary = summarize_cross_validation(
    validation_model,
    x_train,
    y_train,
    n_splits=5,
    random_state=42,
)
print(cv_summary.round(3).to_string(index=False))

plot_cross_validation_metrics(
    cv_summary,
    FIGURES_DIR / "validation_cross_validation_metrics.png",
)

           metric  mean   std
         accuracy 0.855 0.027
        precision 0.873 0.069
           recall 0.810 0.069
               f1 0.837 0.029
          roc_auc 0.902 0.017
average_precision 0.899 0.025


## Final Fit and Held-Out Test Evaluation

After cross-validation, one fresh pipeline is fitted on the full training set and evaluated once on the held-out test set.

In [5]:
final_model = create_logistic_regression_pipeline(
    feature_columns,
    random_state=42,
)
final_model.fit(x_train, y_train)
test_probabilities = final_model.predict_proba(x_test)[:, 1]
test_predictions_050 = (test_probabilities >= 0.50).astype(int)
test_metrics = compute_classification_metrics(
    y_test,
    test_predictions_050,
    test_probabilities,
)
comparison = cv_summary.set_index("metric")[["mean", "std"]].copy()
comparison["held_out_test"] = pd.Series(test_metrics)
print(comparison.round(3).to_string())

plot_confusion_matrix(
    y_test,
    test_predictions_050,
    FIGURES_DIR / "validation_test_confusion_matrix_threshold_050.png",
    title="Held-Out Confusion Matrix at Threshold 0.50",
)

                    mean    std  held_out_test
metric                                        
accuracy           0.855  0.027          0.869
precision          0.873  0.069          0.812
recall             0.810  0.069          0.929
f1                 0.837  0.029          0.867
roc_auc            0.902  0.017          0.966
average_precision  0.899  0.025          0.963


## Calibration Assessment

Calibration asks whether predicted probabilities align with observed outcome frequencies. The small held-out sample makes this estimate uncertain, so the curve is descriptive rather than confirmatory.

In [6]:
brier_score = brier_score_loss(y_test, test_probabilities)
observed_fraction, mean_probability = calibration_curve(
    y_test,
    test_probabilities,
    n_bins=8,
    strategy="quantile",
)
calibration_table = pd.DataFrame(
    {
        "mean_predicted_probability": mean_probability,
        "observed_positive_fraction": observed_fraction,
    }
)
print(f"Held-out Brier score: {brier_score:.3f}")
print(calibration_table.round(3).to_string(index=False))

plot_calibration_assessment(
    y_test,
    test_probabilities,
    FIGURES_DIR / "validation_calibration_curve.png",
    n_bins=8,
)

Held-out Brier score: 0.083
 mean_predicted_probability  observed_positive_fraction
                      0.027                       0.000
                      0.061                       0.000
                      0.181                       0.000
                      0.416                       0.375
                      0.703                       0.429
                      0.865                       0.875
                      0.956                       1.000
                      0.983                       1.000


## Exploratory Threshold Analysis

Changing the probability threshold changes the balance among false positives, false negatives, precision, recall, and specificity. A threshold is therefore not clinically neutral: choosing one requires an explicitly defined use case, error costs, prevalence context, and external validation. This notebook does not select a clinical threshold.

In [7]:
thresholds = [0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80]
threshold_table = threshold_metrics_table(
    y_test,
    test_probabilities,
    thresholds,
)
print(threshold_table.round(3).to_string(index=False))

plot_threshold_metrics(
    threshold_table,
    FIGURES_DIR / "validation_threshold_metrics.png",
)

 threshold  accuracy  precision  recall  specificity    f1  predicted_positive_rate  true_negatives  false_positives  false_negatives  true_positives
       0.2     0.787      0.683   1.000        0.606 0.812                    0.672              20               13                0              28
       0.3     0.820      0.730   0.964        0.697 0.831                    0.607              23               10                1              27
       0.4     0.885      0.818   0.964        0.818 0.885                    0.541              27                6                1              27
       0.5     0.869      0.812   0.929        0.818 0.867                    0.525              27                6                2              26
       0.6     0.885      0.862   0.893        0.879 0.877                    0.475              29                4                3              25
       0.7     0.902      0.893   0.893        0.909 0.893                    0.459              30 

## Cautious Interpretation

Cross-validation describes internal stability within the training partition, while the held-out test set provides one independent evaluation for this split. Similar values can support confidence in workflow consistency, but they do not establish transportability or clinical validity. Calibration and threshold behavior remain uncertain because the held-out sample is small.

## Limitations and Next Steps

- The dataset is historical, small, and from a limited setting.
- Validation is internal only; there is no external cohort.
- Calibration estimates are sensitive to sample size and binning.
- Threshold comparisons use the held-out test set for descriptive sensitivity analysis only; no threshold is selected.
- Future work should prioritize external validation, calibration uncertainty, and decision-context definition before considering more complex models.